# Transfer Learning and Transformers

Training a deep network from scratch on ImageNet takes weeks on multiple GPUs. Training a large language model costs millions of dollars. Yet practitioners routinely achieve state-of-the-art results on small datasets and specialized tasks. **Transfer learning** makes this possible by reusing knowledge learned on large source tasks.

Meanwhile, the **Transformer** architecture — built entirely on attention mechanisms — has become the foundation of modern language models, vision models, and multimodal systems. This notebook covers how transfer learning works in practice, derives the Transformer from first principles, and implements core attention components in NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. What Is Transfer Learning?

**Transfer learning** reuses a model trained on a large **source task** (e.g., ImageNet classification, web-scale text) to solve a smaller **target task** (e.g., detecting a rare disease in X-rays, classifying customer reviews).

The intuition: lower layers of a vision network learn universal features — edges, textures, shapes — that apply across image domains. Higher layers learn task-specific combinations. A model pre-trained on millions of natural images already knows how to see; you only need to teach it your specific categories.

**When transfer learning helps most:**
- Target dataset is **small** (hundreds to thousands of examples).
- Source and target domains are **related** (both are images, both are English text).
- Training from scratch would **overfit** or require impractical compute.

**When it helps less:**
- Target domain is very different from source (satellite imagery vs faces — still often works, but less so).
- You have millions of in-domain labeled examples and can train from scratch.
- Tabular data rarely benefits from neural network transfer learning.

---

## 2. Why Transfer Learning Works

Deep networks learn a **hierarchy of representations**. Early layers capture low-level patterns shared across tasks; later layers specialize.

```
Pre-trained network:
  Layer 1–3:  edges, corners, colors     ← transferable
  Layer 4–6:  textures, parts             ← mostly transferable
  Layer 7+:   task-specific concepts      ← replace or fine-tune
```

Starting from pre-trained weights places optimization in a **good region** of parameter space — near solutions that already encode useful structure. Fine-tuning requires far fewer examples and epochs than random initialization.

This is analogous to how humans learn: knowledge of one programming language makes learning another easier because underlying concepts transfer.

---

## 3. Transfer Learning Strategies

| Strategy | What You Do | When to Use |
|----------|-------------|-------------|
| **Feature extraction** | Freeze pre-trained layers; train only a new classifier head | Very small dataset; similar domain |
| **Fine-tuning** | Unfreeze some or all layers; train with a small learning rate | Moderate dataset; need adaptation |
| **Full fine-tuning** | Train entire network end-to-end | Large enough target data; domain shift |
| **Linear probing** | Train only final linear layer (diagnostic) | Measure representation quality |

**Typical workflow:**
1. Load pre-trained model (ResNet, BERT, etc.).
2. Remove the original classification head.
3. Attach a new head matching your number of classes.
4. Freeze backbone, train head until convergence.
5. Optionally unfreeze top layers and fine-tune with a lower learning rate (often 10× smaller than head training).

**Learning rate discipline:** The pre-trained backbone already has good weights. Large learning rates destroy them. Use a small rate (e.g., $10^{-5}$ to $10^{-4}$) for fine-tuning and a larger rate (e.g., $10^{-3}$) only for the new head.

In [ ]:
# Transfer learning simulation on digit classification
digits = load_digits()
X, y = digits.data / 16.0, digits.target  # normalize pixels to [0, 1]

# Simulate: large source dataset (pre-training) vs small target dataset
X_source, X_target, y_source, y_target = train_test_split(
    X, y, test_size=0.15, random_state=42
)
X_target_tr, X_target_te, y_target_tr, y_target_te = train_test_split(
    X_target, y_target, test_size=0.5, random_state=42
)

# "Pre-train" a feature extractor on the large source set
pretrained = MLPClassifier(
    hidden_layer_sizes=(128, 64), max_iter=500, random_state=42
)
pretrained.fit(X_source, y_source)

def extract_features(model, X):
    """Extract penultimate-layer activations (frozen backbone output)."""
    h = X
    for i in range(len(model.coefs_) - 1):
        h = h @ model.coefs_[i] + model.intercepts_[i]
        if model.activation == "relu":
            h = np.maximum(0, h)
        elif model.activation == "tanh":
            h = np.tanh(h)
        else:
            h = 1 / (1 + np.exp(-np.clip(h, -500, 500)))
    return h

X_tr_feat = extract_features(pretrained, X_target_tr)
X_te_feat = extract_features(pretrained, X_target_te)

scaler = StandardScaler().fit(X_tr_feat)
X_tr_feat_s = scaler.transform(X_tr_feat)
X_te_feat_s = scaler.transform(X_te_feat)

# Strategy 1: Train only a linear head on frozen features (feature extraction)
head = LogisticRegression(max_iter=1000, random_state=42)
head.fit(X_tr_feat_s, y_target_tr)
acc_transfer = accuracy_score(y_target_te, head.predict(X_te_feat_s))

# Baseline: Train full network from scratch on the small target set
scratch = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
scratch.fit(X_target_tr, y_target_tr)
acc_scratch = accuracy_score(y_target_te, scratch.predict(X_target_te))

print(f"Target set size: {len(X_target_tr)} train, {len(X_target_te)} test")
print(f"Pre-trained backbone + linear head:  {acc_transfer:.3f}")
print(f"Train from scratch on small set:     {acc_scratch:.3f}")

In [ ]:
# Visualize: transfer learning closes the gap on small data
sizes = [20, 50, 100, 200]
acc_transfer_list, acc_scratch_list = [], []

for n in sizes:
    idx = np.random.choice(len(X_target), n, replace=False)
    X_sub, y_sub = X_target[idx], y_target[idx]
    X_sub_tr, X_sub_te, y_sub_tr, y_sub_te = train_test_split(
        X_sub, y_sub, test_size=0.3, random_state=42
    )
    feat_tr = extract_features(pretrained, X_sub_tr)
    feat_te = extract_features(pretrained, X_sub_te)
    sub_scaler = StandardScaler().fit(feat_tr)
    feat_tr_s = sub_scaler.transform(feat_tr)
    feat_te_s = sub_scaler.transform(feat_te)
    h = LogisticRegression(max_iter=1000, random_state=42)
    h.fit(feat_tr_s, y_sub_tr)
    acc_transfer_list.append(accuracy_score(y_sub_te, h.predict(feat_te_s)))
    s = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
    s.fit(X_sub_tr, y_sub_tr)
    acc_scratch_list.append(accuracy_score(y_sub_te, s.predict(X_sub_te)))

plt.figure(figsize=(8, 4))
plt.plot(sizes, acc_transfer_list, "o-", label="Transfer (frozen features + head)", linewidth=2)
plt.plot(sizes, acc_scratch_list, "s-", label="From scratch", linewidth=2)
plt.xlabel("Training examples")
plt.ylabel("Test accuracy")
plt.title("Transfer learning wins with limited target data")
plt.legend()
plt.show()

---

## 4. Pre-trained Models in Practice

Modern ecosystems provide ready-to-use pre-trained models:

| Domain | Popular Models | Typical Source Task |
|--------|---------------|--------------------|
| Vision | ResNet, EfficientNet, ViT | ImageNet classification |
| Language | BERT, RoBERTa, GPT | Masked LM / next-token prediction |
| Multimodal | CLIP, LLaVA | Image-text alignment |
| Speech | Whisper | Speech recognition |

**Model hubs** (Hugging Face, torchvision, timm) host thousands of checkpoints with standardized APIs. Loading a pre-trained ResNet or BERT is often one line of code.

For vision: replace the final fully connected layer (`model.fc` in ResNet) with `nn.Linear(in_features, num_classes)`. For BERT: attach a classification head on the `[CLS]` token representation.

---

## 5. The Limits of Recurrence

RNNs and LSTMs process sequences one token at a time, maintaining a hidden state. This creates two fundamental problems:

1. **Sequential computation** — step $t$ cannot begin until step $t-1$ finishes. No parallelization across time steps during training.
2. **Information bottleneck** — compressing an entire source sentence into a single fixed-size vector (in early seq2seq models) loses information for long sequences.

**Attention** (Bahdanau, 2015) addressed the bottleneck by letting the decoder look at all encoder states. **Self-attention** (Vaswani et al., 2017) went further: remove recurrence entirely. Every token directly attends to every other token in parallel.

The result — the **Transformer** — trains faster on GPUs, captures long-range dependencies directly, and scales to billions of parameters.

---

## 6. Attention Intuition

When reading the sentence *"The animal didn't cross the street because it was too tired,"* understanding *"it"* requires attending to *"animal"* — not *"street."*

**Attention** computes a weighted sum of values, where weights reflect relevance:

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right) \mathbf{V}$$

- **Query (Q)** — what am I looking for? ("I need to resolve 'it'")
- **Key (K)** — what do I offer? (each word's identity for matching)
- **Value (V)** — what information do I carry? (each word's representation)

High attention weight between query *"it"* and key *"animal"* means the model pulls information from *"animal"* when processing *"it."*

In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - x.max(axis=axis, keepdims=True))
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """Q, K, V: (seq_len, d_k)"""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)          # (seq_len, seq_len)
    weights = softmax(scores, axis=-1)
    output = weights @ V                    # (seq_len, d_v)
    return output, weights

# Toy example: 4 tokens, 8-dimensional embeddings
seq_len, d_model = 4, 8
X = np.random.randn(seq_len, d_model)

# In self-attention, Q, K, V are linear projections of the same input
W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q, K, V = X @ W_q, X @ W_k, X @ W_v
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Input shape:  {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"\nAttention weights (rows=query, cols=key):")
print(attn_weights)

In [ ]:
# Visualize attention weight matrix
tokens = ["The", "cat", "sat", "down"]

plt.figure(figsize=(6, 5))
plt.imshow(attn_weights, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Attention weight")
plt.xticks(range(len(tokens)), tokens)
plt.yticks(range(len(tokens)), tokens)
plt.xlabel("Key (attended to)")
plt.ylabel("Query (attending from)")
plt.title("Self-attention weight matrix")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        plt.text(j, i, f"{attn_weights[i,j]:.2f}", ha="center", va="center",
                 color="white" if attn_weights[i,j] > 0.5 else "black", fontsize=9)
plt.tight_layout()
plt.show()

---

## 7. Why Scale by $\sqrt{d_k}$?

Dot products $\mathbf{q} \cdot \mathbf{k}$ grow in magnitude as dimension $d_k$ increases. Large dot products push softmax into regions with extremely small gradients — the distribution becomes nearly one-hot, and learning stalls.

Dividing by $\sqrt{d_k}$ keeps the variance of dot products stable regardless of dimension, maintaining healthy softmax gradients.

For $d_k = 64$: divide scores by 8. For $d_k = 512$: divide by $\approx 22.6$.

---

## 8. Multi-Head Attention

A single attention head learns one type of relationship. **Multi-head attention** runs $h$ parallel attention operations with separate projection matrices, then concatenates the results:

$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q, \mathbf{K}\mathbf{W}_i^K, \mathbf{V}\mathbf{W}_i^V)$$

$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \mathbf{W}^O$$

Different heads can specialize:
- Syntactic relationships (subject–verb agreement).
- Positional proximity (attending to neighboring words).
- Semantic similarity (synonyms, coreference).

Typical configurations: 8–16 heads, each with $d_k = d_{\text{model}} / h$.

In [ ]:
def multi_head_attention(X, d_model, num_heads):
    """Simplified multi-head self-attention."""
    assert d_model % num_heads == 0
    d_k = d_model // num_heads
    seq_len = X.shape[0]
    heads = []
    all_weights = []

    for h in range(num_heads):
        W_q = np.random.randn(d_model, d_k) * np.sqrt(2.0 / d_model)
        W_k = np.random.randn(d_model, d_k) * np.sqrt(2.0 / d_model)
        W_v = np.random.randn(d_model, d_k) * np.sqrt(2.0 / d_model)
        Q, K, V = X @ W_q, X @ W_k, X @ W_v
        out, w = scaled_dot_product_attention(Q, K, V)
        heads.append(out)
        all_weights.append(w)

    concat = np.concatenate(heads, axis=-1)  # (seq_len, d_model)
    W_o = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
    return concat @ W_o, all_weights

d_model, num_heads = 16, 4
X = np.random.randn(5, d_model)
mha_out, head_weights = multi_head_attention(X, d_model, num_heads)

print(f"Input:  {X.shape}")
print(f"Output: {mha_out.shape}")
print(f"Number of attention heads: {num_heads}")
print(f"Each head attention matrix: {head_weights[0].shape}")

---

## 9. Positional Encoding

Self-attention is **permutation invariant** — shuffling token order does not change the pairwise dot products (only which row is which). But word order matters in language. **Positional encodings** inject position information.

The original Transformer uses fixed sinusoidal encodings:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

These are **added** to token embeddings before the first attention layer. Modern models often use **learned** positional embeddings or **rotary position embeddings (RoPE)** — but the purpose is the same: tell the model where each token sits in the sequence.

In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

pe = sinusoidal_positional_encoding(50, 64)

plt.figure(figsize=(10, 4))
plt.imshow(pe, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.colorbar(label="Encoding value")
plt.xlabel("Embedding dimension")
plt.ylabel("Position in sequence")
plt.title("Sinusoidal positional encodings")
plt.show()

---

## 10. The Transformer Block

A single Transformer layer combines several components:

```
Input embeddings + positional encoding
        ↓
  Multi-Head Self-Attention
        ↓
  Add & Norm (residual + layer norm)
        ↓
  Feed-Forward Network (2 linear layers + ReLU/GELU)
        ↓
  Add & Norm (residual + layer norm)
        ↓
  Output to next layer
```

**Residual connections** — $\mathbf{x} + \text{SubLayer}(\mathbf{x})$ — let gradients flow directly through the network, enabling very deep stacks (12, 24, 96+ layers).

**Layer normalization** — normalizes across features within each token (not across the batch like batch norm). Applied after the residual add in modern implementations (Post-LN) or before the sublayer (Pre-LN, more stable for deep models).

**Feed-forward network (FFN)** — two linear transformations with a nonlinearity in between, applied independently to each token position:

$$\text{FFN}(\mathbf{x}) = \text{GELU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

The FFN is typically 4× wider than $d_{\text{model}}$ (e.g., 768 → 3072 → 768 in BERT-base).

In [ ]:
def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    sigma = x.std(axis=-1, keepdims=True)
    return (x - mu) / (sigma + eps)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def feed_forward(x, d_model, d_ff):
    W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
    W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
    return gelu(x @ W1) @ W2

def transformer_block(x, d_model, num_heads, d_ff):
    # Self-attention sublayer with residual
    attn_out, _ = multi_head_attention(x, d_model, num_heads)
    x = layer_norm(x + attn_out)
    # Feed-forward sublayer with residual
    ff_out = feed_forward(x, d_model, d_ff)
    x = layer_norm(x + ff_out)
    return x

d_model, num_heads, d_ff = 32, 4, 128
seq_len = 6
x = np.random.randn(seq_len, d_model)

for layer in range(3):
    x = transformer_block(x, d_model, num_heads, d_ff)

print(f"After 3 Transformer blocks: {x.shape}")
print(f"Output mean: {x.mean():.4f}, std: {x.std():.4f}")

---

## 11. Transformer Architectures

The original Transformer has an **encoder** and **decoder**. Modern models use one or the other:

| Architecture | Structure | Pre-training Task | Examples |
|-------------|-----------|-------------------|----------|
| **Encoder-only** | Bidirectional self-attention | Masked language modeling | BERT, RoBERTa |
| **Decoder-only** | Causal (left-to-right) self-attention | Next-token prediction | GPT, LLaMA |
| **Encoder-decoder** | Encoder + cross-attention decoder | Seq2seq denoising | T5, BART, original Transformer |

**Encoder-only (BERT):** Sees the full context in both directions. Excellent for classification, named entity recognition, and understanding tasks. Add a task-specific head on top of the `[CLS]` token or token representations.

**Decoder-only (GPT):** Each token attends only to previous tokens (causal mask). Trained to predict the next token. Excellent for generation — text completion, code, dialogue. Scales to trillions of parameters.

**Encoder-decoder (T5):** Encoder processes input; decoder generates output with cross-attention to encoder states. Natural for translation, summarization, and question answering.

---

## 12. Causal (Masked) Attention

Decoder-only models must not peek at future tokens during training. A **causal mask** sets attention scores for future positions to $-\infty$ before softmax:

$$\text{scores}_{ij} = \begin{cases} \frac{\mathbf{q}_i \cdot \mathbf{k}_j}{\sqrt{d_k}} & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

This ensures token $i$ only attends to tokens at positions $\leq i$. Combined with next-token prediction loss, the model learns to generate text autoregressively.

In [ ]:
def causal_attention(Q, K, V):
    d_k = Q.shape[-1]
    seq_len = Q.shape[0]
    scores = Q @ K.T / np.sqrt(d_k)
    # Causal mask: upper triangle = -inf
    mask = np.triu(np.ones((seq_len, seq_len)) * -1e9, k=1)
    scores = scores + mask
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

seq_len = 5
d_k = 8
Q = K = V = np.random.randn(seq_len, d_k)
_, causal_weights = causal_attention(Q, K, V)

plt.figure(figsize=(5, 4))
plt.imshow(causal_weights, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Weight")
plt.title("Causal attention mask (decoder-only)")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.show()

---

## 13. Fine-Tuning Transformers

Pre-trained language models encode rich linguistic knowledge. Fine-tuning adapts them to specific tasks:

**Text classification (BERT-style):**
1. Add `[CLS]` token at the start of each input.
2. Pass through frozen or fine-tuned BERT.
3. Take `[CLS]` output → linear layer → class logits.
4. Train with cross-entropy on labeled examples.

**Text generation (GPT-style):**
1. Provide a prompt.
2. Model predicts next token; append it to the sequence.
3. Repeat until end-of-sequence token or max length.

**Parameter-efficient fine-tuning (PEFT):** Methods like **LoRA** (Low-Rank Adaptation) train small adapter matrices instead of the full model. Fine-tune a 7B-parameter model on a single GPU by updating only ~0.1% of parameters.

| Method | What Gets Trained | Memory | Use Case |
|--------|------------------|--------|----------|
| Full fine-tuning | All parameters | High | Large target dataset |
| Linear probing | Classification head only | Low | Quick baseline |
| LoRA | Low-rank adapters | Medium | Limited GPU memory |
| Prompt tuning | Soft prompt embeddings | Very low | Few-shot adaptation |

---

## 14. Vision Transformers (ViT)

Transformers are not limited to text. **Vision Transformer (ViT)** splits an image into fixed-size patches, flattens each patch into a vector, and treats patches as a sequence of tokens — just like words in a sentence.

```
Image (224×224) → 16×16 patches → 196 patch tokens + [CLS] token
        ↓
  Linear embedding + positional encoding
        ↓
  Transformer encoder (12 layers)
        ↓
  [CLS] token → classification head
```

ViT requires large datasets or strong pre-training to outperform CNNs on smaller data. Hybrid approaches and pre-training on ImageNet-21k make ViT practical. Transfer learning from pre-trained ViT models follows the same freeze/fine-tune workflow as CNNs.

---

## 15. Why Transformers Dominate

| Property | RNN/LSTM | CNN | Transformer |
|----------|----------|-----|-------------|
| **Parallelism** | Sequential | High | High |
| **Long-range deps** | Limited (vanishing gradients) | Grows with depth | Direct (any token pair) |
| **Scalability** | Poor beyond ~1B params | Good for vision | Excellent (100B+ params) |
| **Transfer learning** | Moderate | Strong (vision) | Strong (language + vision) |
| **Interpretability** | Hidden state opaque | Feature maps | Attention weights visualizable |

Transformers benefit from **scaling laws**: performance improves predictably as model size, data, and compute increase. This motivated the race to ever-larger language models — GPT-3 (175B), PaLM (540B), GPT-4 (rumored trillions of parameters in mixture-of-experts form).

The same architecture powers chatbots, code assistants, image generators (when combined with diffusion), protein structure prediction, and more.

---

## 16. Practical Training Recipe

A modern fine-tuning workflow:

1. **Choose a pre-trained model** matched to your task (BERT for classification, GPT for generation).
2. **Prepare data** — tokenize with the model's tokenizer; keep train/val splits.
3. **Start with linear probing** — freeze backbone, train head only. Establishes a baseline.
4. **Fine-tune** — unfreeze top layers (or use LoRA); learning rate $10^{-5}$ to $10^{-4}$.
5. **Monitor validation loss** — early stopping prevents overfitting on small datasets.
6. **Evaluate** on held-out test set once.

**Common pitfalls:**
- Learning rate too high → catastrophic forgetting of pre-trained knowledge.
- Tokenizer mismatch → always use the tokenizer that matches the pre-trained model.
- Sequence length truncation → long documents may lose important context; consider chunking or sliding windows.
- Not enough epochs on small data → monitor val loss; small datasets converge quickly.

---

## 17. Summary

**Transfer learning** reuses representations learned on large source tasks to solve smaller target tasks efficiently. Freeze the backbone and train a new head for very small datasets; fine-tune with a low learning rate when more adaptation is needed.

The **Transformer** replaces recurrence with **self-attention** — every token attends to every other token in parallel. **Multi-head attention** captures diverse relationships. **Positional encodings** inject order. **Residual connections** and **layer normalization** enable deep stacks.

Encoder-only models (BERT) excel at understanding; decoder-only models (GPT) excel at generation; encoder-decoder models (T5) handle seq2seq tasks. **Fine-tuning** and **parameter-efficient methods** like LoRA make these billion-parameter models accessible on modest hardware.

Together, transfer learning and Transformers form the foundation of modern AI — from classifying medical images with a pre-trained ResNet to building conversational agents on top of large language models.